# Active Matter JEPA: Global vs Spatial Latents

This notebook runs the current `train_active_matter_jepa.py` pipeline for the controlled global-vs-spatial JEPA comparison. The training code lives in the `.py` file so the notebook stays in sync with the restartable script used on the cluster.


## 1. Setup

Edit the paths in the next cell only if your project or dataset is somewhere different. By default, the notebook uses `/scratch/$USER/data` when available and writes outputs under `/scratch/$USER/am_jepa`.


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

PROJECT_DIR = Path.cwd()
USER = os.environ.get("USER", "sp9002")

def first_existing(*candidates: Path) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]

DATA_ROOT = Path(os.environ.get("DATA_ROOT", first_existing(
    Path(f"/scratch/{USER}/data"),
    PROJECT_DIR / "data",
)))
OUTPUT_BASE = Path(os.environ.get("OUTPUT_BASE", (
    Path(f"/scratch/{USER}/am_jepa") if Path(f"/scratch/{USER}").exists() else PROJECT_DIR / "outputs" / "notebook_am_jepa"
)))
WANDB_MODE = os.environ.get("WANDB_MODE", "disabled")
SMOKE_TEST = os.environ.get("SMOKE_TEST", "0") == "1"

GLOBAL_OUTPUT_DIR = OUTPUT_BASE / "global_run"
SPATIAL_OUTPUT_DIR = OUTPUT_BASE / "spatial_run"
for directory in (GLOBAL_OUTPUT_DIR, SPATIAL_OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(json.dumps({
    "project_dir": str(PROJECT_DIR),
    "data_root": str(DATA_ROOT),
    "global_output_dir": str(GLOBAL_OUTPUT_DIR),
    "spatial_output_dir": str(SPATIAL_OUTPUT_DIR),
    "wandb_mode": WANDB_MODE,
    "smoke_test": SMOKE_TEST,
}, indent=2))


## 2. Run Training

This launches the same script used by Slurm. It trains/evaluates the global run first, then the spatial run. Checkpoints and final probe outputs are written into the output directories above.


In [ ]:
def run_jepa(config_path: str, model_type: str, output_dir: Path) -> None:
    cmd = [
        sys.executable,
        str(PROJECT_DIR / "train_active_matter_jepa.py"),
        "--config",
        str(PROJECT_DIR / config_path),
        "--model_type",
        model_type,
        "--data_root",
        str(DATA_ROOT),
        "--output_dir",
        str(output_dir),
        "--wandb_mode",
        WANDB_MODE,
    ]
    if SMOKE_TEST:
        cmd.append("--smoke_test")
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=PROJECT_DIR, check=True)

run_jepa("configs/global.yaml", "global", GLOBAL_OUTPUT_DIR)
run_jepa("configs/spatial.yaml", "spatial", SPATIAL_OUTPUT_DIR)


## 3. Training Curves


In [ ]:
def read_history(output_dir: Path, model_name: str) -> pd.DataFrame:
    path = output_dir / "training_history.csv"
    df = pd.read_csv(path)
    df["model"] = model_name
    return df

history_df = pd.concat([
    read_history(GLOBAL_OUTPUT_DIR, "global"),
    read_history(SPATIAL_OUTPUT_DIR, "spatial"),
], ignore_index=True)

fig, ax = plt.subplots(figsize=(7, 4))
for model_name, group in history_df.groupby("model"):
    ax.plot(group["epoch"], group["train_loss"], marker="o", label=f"{model_name} train")
    ax.plot(group["epoch"], group["valid_loss"], marker="s", label=f"{model_name} valid")
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.legend()
ax.set_title("JEPA training curves")
plt.show()

history_df.tail()


## 4. Frozen Probe Results


In [ ]:
def read_probe_results(output_dir: Path, model_name: str) -> pd.DataFrame:
    path = output_dir / "frozen_probe_results.csv"
    df = pd.read_csv(path)
    df["model"] = model_name
    return df

results_df = pd.concat([
    read_probe_results(GLOBAL_OUTPUT_DIR, "global"),
    read_probe_results(SPATIAL_OUTPUT_DIR, "spatial"),
], ignore_index=True)

headline_df = results_df[(results_df["split"] == "test") & (results_df["probe"] == "linear")][
    ["model", "alpha_mse", "zeta_mse"]
].reset_index(drop=True)

print(headline_df.to_markdown(index=False))
headline_df


## 5. Predicted vs True Zeta


In [ ]:
for model_name, output_dir in [("global", GLOBAL_OUTPUT_DIR), ("spatial", SPATIAL_OUTPUT_DIR)]:
    image_path = output_dir / "figures" / "predicted_vs_true_zeta_linear.png"
    print(model_name, image_path)
    if image_path.exists():
        display(Image(filename=str(image_path)))
    else:
        print(f"Missing figure: {image_path}")


## 6. Final Summaries


In [ ]:
summaries = {}
for model_name, output_dir in [("global", GLOBAL_OUTPUT_DIR), ("spatial", SPATIAL_OUTPUT_DIR)]:
    summary_path = output_dir / "final_summary.json"
    summaries[model_name] = json.loads(summary_path.read_text()) if summary_path.exists() else {"missing": str(summary_path)}

summaries
